# Evaluate Gemini3 Model

In [ ]:
# Installing

!pip install litellm==1.73.6

In [ ]:
# importing

import os
from pathlib import Path

from google.colab import userdata
from os.path import join

from tqdm import tqdm

import litellm
from litellm import completion

import json

In [ ]:
data_dir = "/gdrive/MyDrive/VLMs-Dataset"
sample_image_path = f"{data_dir}/0001/page_006.jpg"

openrouter_token = userdata.get('openrouter')
os.environ["OPENAI_API_KEY"] = openrouter_token
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
cloud_model_id = "openai/google/gemini-3.1-flash-lite-preview" # adding openai/ for the scentence to make the llm understand that model is compatable with openai


In [ ]:
chat_prompt = """
{
"document_classification": {
"type": "enum: official_letter | decree | regulation | statistical_report | table_of_contents | administrative_decision | other",
"subtype": "string or null",
"category": "enum: legal | administrative | financial | statistical | correspondence | technical | hr | other",
"primary_language": "enum: arabic | english | french | mixed | other",
"secondary_languages": ["array of language codes if multilingual"]
},

"source": {
"issuing_authority": "string: full organization name in ORIGINAL SCRIPT",
"department": "string or null",
"location": "string or null",
"document_number": "string or null EXACTLY as shown",
"related_references": ["array of all document numbers mentioned anywhere"],
"page_number": "string or null EXACTLY as shown",
"total_pages": "string or null if mentioned",
"confidentiality_level": "string or null (e.g., سري، عاجل، داخلي)"
},

"dates": {
"primary_date": {
"date_text": "string EXACTLY as written",
"calendar_type": "enum: hijri | gregorian | unknown",
"date_type": "enum: issue_date | effective_date | received_date | other",
"indicators": "string like هـ or م or null",
"location_in_document": "enum: header | body | footer | stamp | margin | other"
},
"additional_dates": [
{
"date_text": "string EXACTLY as written",
"calendar_type": "enum: hijri | gregorian | unknown",
"date_type": "enum: reference_date | deadline | expiry_date | effective_date | other",
"context": "brief explanation of why this date appears",
"indicators": "string or null",
"location_in_document": "enum: header | body | footer | stamp | margin | other"
}
]
},

"people": [
{
"name": "string EXACTLY as written",
"title": "string or null",
"role": "enum: sender | recipient | signatory | mentioned | other",
"signature_present": "boolean"
}
],

"stamps": [
{
"text": "string EXACTLY as visible",
"shape": "string or null (e.g., circular, rectangular)",
"color": "string or null",
"location_in_document": "enum: header | body | footer | margin | other"
}
],

"signatures": [
{
"name": "string or null",
"title": "string or null",
"location_in_document": "enum: header | body | footer | margin | other"
}
],

"content_structure": {
"has_articles": "boolean",
"articles": [
{
"article_number": "string EXACTLY as written",
"article_title": "string or null",
"article_text": "string EXACTLY as written"
}
],
"has_numbered_paragraphs": "boolean",
"numbered_paragraphs": [
{
"number": "string EXACTLY as written",
"text": "string EXACTLY as written"
}
]
},

"full_text": "string: COMPLETE text of the document in reading order exactly as written"
}
""".strip()

In [ ]:
import base64

def image_to_base64_data_uri(image_path): # convert image to base64 data uri for apis
  with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode('utf-8')

  # determine image type from extension:
  ext = image_path.lower().split('.')[-1]
  mime_type = f"image/{ext}" if ext == "jpg" else "image/jpeg"

  return f"data:image/{mime_type};base64,{image_base64}"


In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text","text": chat_prompt}, # prompt
             {"type": "image_url",
             "image_url":{
                "url":image_to_base64_data_uri(sample_image_path)
                },
            }
      ]
    }
]

response = completion(
    model=cloud_model_id,
    messages=messages,
    max_tokens=4096,

)


# knowledge Distillation
**Accepting the response of Gemini model** so now we will pass all pic for making

In [ ]:
# Mount to the drive
from google.colab import drive
drive.mount('/gdrive')

data_dir = "/gdrive/MyDrive/VLMs-Dataset"

Mounted at /gdrive


In [ ]:
##### Creating a dic of folders name as a key and the images name as a values
import os
from pathlib import Path

# Set your base directory containing all the folders
base_dir = data_dir  # Change this to your base directory path

# Supported image extensions
image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.tif', '.webp', '.svg'}

# Create dictionary {folder_name: [list of image paths]}
images_dict = {}

for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)

    if os.path.isdir(folder_path):  # only process directories
        images = []
        for file in os.listdir(folder_path):
            if Path(file).suffix.lower() in image_extensions:
                images.append(os.path.join(folder_path, file))

        images_dict[folder_name] = images

# check it worked :
print(images_dict["0056"])

['/gdrive/MyDrive/VLMs-Dataset/0056/page_005.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_006.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_002.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_003.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_007.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_004.jpg', '/gdrive/MyDrive/VLMs-Dataset/0056/page_001.jpg']


In [ ]:

output_dir = f"{data_dir}/pdf_images" # file that contain all images
pdf_images_paths = images_dict  # copy of the dictionary


In [ ]:
cloud_model_id = "openai/google/gemini-3.1-flash-lite-preview" # adding openai/ for the scentence to make the llm understand that model is compatable with openai
output_sft_file = join(output_dir, "ocr-image-sft.jsonl")

price_per_1m_input_tokens = 0.5
price_per_1m_output_tokens = 3.0

prompt_tokens = 0
completion_tokens = 0
total_tokens = 0
total_cost = 0
ix = 0

for pdf_name, images in pdf_images_paths.items():
    ix+=1

    ## if it break in the midlle and i want to continue from the break
    # if ix < 500:
    #   continue
    for image_path in tqdm(images,desc=f"file_{pdf_name}" ):
        messages = [
        {
        "role": "user",
        "content": [
            {"type": "text","text": chat_prompt}, # prompt
             {"type": "image_url",
             "image_url":{
                "url":image_to_base64_data_uri(image_path)},
                 }
              ]
          }
       ]

        response = completion(
              model=cloud_model_id,
              messages=messages,
              max_tokens=8000,

          )
        if response.choices[0].finish_reason != 'stop':
            print(f"Error Stop Reason: {ix}", response.choices[0].finish_reason)
            continue


llm_response = response.choices[0].message.content

with open(output_sft_file, "a", encoding="utf-8") as f:
  f.write(json.dumps(
      {
    "id":ix,
    "pdf_name":pdf_name,
    "image_path":image_path,
    "output":llm_response,
      },
      default=str,
      ensure_ascii=False,

    )+"/n",)

  completion_tokens += response.usage.completion_tokens
  prompt_tokens += response.usage.prompt_tokens

  if(ix % 3) ==0:
    cost_input = (prompt_tokens/1000000) * price_per_1m_input_tokens
    cost_output = (completion_tokens/1000000) * price_per_1m_output_tokens
    total_cost += cost_input + cost_output

    print(f"Iteration {ix}: Total Cost = ${total_cost:.4f}")

file_0053:  30%|██▉       | 20/67 [01:39<03:41,  4.71s/it]

Error Stop Reason: 1 error


file_0053:  43%|████▎     | 29/67 [02:25<03:10,  5.01s/it]


KeyboardInterrupt: 

# Formating DataSet For FineTuneing (LLamaFactory)
- Open LLama Factory in github and open the template for the model(Gemma 3) and the formatting of the data that needed to use :

In [ ]:
# installing

! pip install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00


In [1]:
# Mount to the drive
from google.colab import drive
drive.mount('/gdrive')


Mounted at /gdrive


In [3]:
# importing

from os.path import join
import json
#import json_repair

In [4]:
data_dir = "/gdrive/MyDrive/VLMs-Dataset"
sample_image_path = f"{data_dir}/0001/page_006.jpg"
output_dir = f"{data_dir}/pdf_images" # file that contain all images
output_sft_file = join(data_dir, "ocr-images-sft.jsonl")
train_ds = join(data_dir, "train.json")
val_ds = join(data_dir, "val.json")

In [ ]:
import json_repair
def parse_json(text):
  try:
    return json_repair.loads(text)
  except:
    return None

In [ ]:
for line in open(output_sft_file):
  print(line)
  break

{"id": 0, "pdf_name": "0001.pdf", "image_path": "/gdrive/MyDrive/youtube-resources/temp/image-ocr-finetune/assets/pdf_images/0001/page_001.jpg", "model_id": "openai/google/gemini-3-flash-preview", "output": "{\n  \"document_classification\": {\n    \"type\": \"administrative_decision\",\n    \"subtype\": \"قرار مجلس الوزراء\",\n    \"category\": \"legal\",\n    \"primary_language\": \"arabic\",\n    \"secondary_languages\": []\n  },\n  \"source\": {\n    \"issuing_authority\": \"المملكة العربية السعودية الأمانة العامة لمجلس الوزراء\",\n    \"department\": \"قرارات مجلس الوزراء\",\n    \"location\": null,\n    \"document_number\": \"728\",\n    \"related_references\": [\n      \"51074\",\n      \"38/338168\",\n      \"1396\",\n      \"661\",\n      \"820\",\n      \"2-17/41/د\",\n      \"37/138\",\n      \"36/175\",\n      \"7637\"\n    ],\n    \"dates\": {\n      \"primary_date\": {\n        \"date_text\": \"1441/11/16هـ\",\n        \"calendar_type\": \"hijri\",\n        \"date_type\":

**we have a problem that the changing between the video and the real doing so this code will not run correctly and we will use the ready files from the video to do finetuning**

In [ ]:
import random
llm_finetuning_data = []

# splitting the output to to cases : one for the content and the other for the description:
# the content :
task_1_message = """

You are a professional OCR Details Extractor.
Your rule to extract: the page markdown content in addition to the structural_elements of the document.
Extract the final output into a json format.
Do not generate any introduction or conclusion.
""".strip()
# the description :
task_2_message = """

You are a professional OCR Details Extractor.
Your rule to extract: the documnet classification, source, physical_properties, official_marks, signatures_authorization, routing.
Extract the final output into a json format.
Do not generate any introduction or conclusion.
""".strip()

# making complete folder for the validation to make sure for a good validation (model not see any image from the folder befor )
val_folders=['0012.pdf','0006.pdf','0070.pdf']

train_ds = []
val_ds = []

# making sure their are not duplications
image_paths_set = set()

for line in open(output_sft_file):

  # clearing empty lines
  if line.strip() == "":
    continue

  # turn to json format
  data = json.loads(line.strip())

  #using the def parse_json to solve the error
  llm_output = parse_json(data['output'])
  if llm_output is None:
    continue

  folder_name = data['pdf_name']


  # making sure their are not duplications :
  if data['image_path'] in image_paths_set:
    continue

  image_paths_set.add(data['image_path'])

  # create the output for the task one (content)
  task_1_output = {
      'content': llm_output['content'],
      'structural_elements': llm_output.get("structural_elements", ""),
      }

  # delete
  del llm_output['content']
  # check the existance :
  if 'structural_elements' in llm_output:
    del llm_output['structural_elements']


  # create the output for the discription
  task_2_output = llm_output

  # tasks format from LLama Factory
  task_1_sft_record = {
    "conversations": [
        {
            "value": "<image>" + task_1_message,
            "from": "human"
        },
        {
            "value": json.dumps(task_1_output, ensure_ascii=False, default=str),
            "from": "gpt"
        }
    ],
    "images": [
        data["image_path"]
    ]  # if it is one image there will be one <image> tag above
}


task_2_sft_record =  {
    "conversations": [
      {
        "value": "<image>"+task_2_message,
        "from": "human"

      },
      {
        "value": json.dumps(task_2_output, ensure_ascii=False, default=str),
        "from": "gpt"

      }
      ],

        "images": [
            data['image_path']
            ] # if it one image so it will be one tag above <image>
  }

if folder_name in val_folders:
  val_ds.append(task_1_sft_record)
  val_ds.append(task_2_sft_record)
else:
  train_ds.append(task_1_sft_record)
  train_ds.append(task_2_sft_record)

random.Random(101).shuffle(train_ds)
random.Random(101).shuffle(val_ds)


In [ ]:
# Error Unterminated string starting : meaning the llm return a foreign character ?
# solving the error is in the lib json-repair

## Fixing Formats

In [ ]:
# Load the file
with open(val_ds, 'r') as f:
    val_ds = json.load(f)

# Get the length
print(len(val_ds))  # Output: 80

80


In [ ]:
# Load the file
with open(train_ds, 'r') as f:
    train_ds = json.load(f)

# Get the length
print(len(train_ds))

3912


In [ ]:
train_ds[0]

{'conversations': [{'value': '<image>You are a professional OCR Details Extractor.\nYour rule to extract: the page markdown content in addition to the structural_elements of the document.\nExtract the final output into a json format.\nDo not generate any introduction or conclusion.',
   'from': 'human'},
  {'value': '{"output": {"subject": "نظام القضاء / لائحة التفتيش القضائي", "subject_translation": null, "keywords": ["التفتيش القضائي", "نظام القضاء", "أعضاء السلك القضائي", "السجل السري", "اللائحة"], "full_text": "بسم الله الرحمن الرحيم\\nالمملكة العربية السعودية\\nالمجلس الأعلى للقضاء\\n(١٥٢)\\nالرقم: ...........\\nالتاريخ: ...........\\nالمرفقات: ...........\\nالموضوع: ...........\\nالمجلس الأعلى للقضاء\\nالتفتيش القضائي\\n\\nالمادة الرابعة والستون :\\nيخضع كتاب العدل للتفتيش القضائي ، وفقاً لأحكام نظام القضاء .\\n\\nالمادة الخامسة والستون :\\n١- يكون لكل عضو من أعضاء السلك القضائي ملف ، يودع فيه تقارير التفتيش ، والتحقيق ، والاعتراضات ، والقرارات ذات الصلة .\\n٢- يكون لكل عضو من أع

In [ ]:
val_ds[0]

{'conversations': [{'value': '<image>You are a professional OCR Details Extractor.\nYour rule to extract the: document_classification, source, physical_properties, official_marks, signatures_authorization, routing_distribution, attachments_references, condition_notes and confidence_quality of the document.\nExtract the final output into a json format.\nDo not generate any introduction or conclusion.',
   'from': 'human'},
  {'value': '{"document_classification": {"type": "regulation", "subtype": "إجرائي", "category": "legal", "primary_language": "arabic", "secondary_languages": []}, "source": {"issuing_authority": "وزارة العدل", "department": null, "location": "المملكة العربية السعودية", "document_number": null, "related_references": [], "dates": {"primary_date": null, "additional_dates": []}}, "physical_properties": {"page_number": "13", "total_pages": null, "image_type": "digital", "quality": "high", "color_mode": "color", "has_watermark": false, "watermark_description": null, "has_s

In [ ]:
val_ds[10]['images'][0]

'/workspace/pdf_images/0011/page_009.jpg'

In [ ]:
 # replacing file image path : ## already done -->

for i in range(len(val_ds)):
  val_ds[i]['images'][0] = val_ds[i]['images'][0].replace("/workspace", "/gdrive/MyDrive/VLMs-Dataset")

for i in range(len(train_ds)):
  train_ds[i]['images'][0] = train_ds[i]['images'][0].replace("/workspace", "/gdrive/MyDrive/VLMs-Dataset")


In [ ]:
# saving the files again :
with open(join(data_dir,"val-v1-edited.json"), "w") as dest:
  json.dump(val_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir,"train_ds-v1-edited.json"), "w") as dest:
  json.dump(train_ds, dest, ensure_ascii=False, default=str)

## llama factory

In [5]:
# installing requirements
# ! pip install transformers optimum datasets torch==2.8.0 torchvision==0.23 torchaudio==2.8.0

# installing llama factory:
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
%cd LlamaFactory
!pip install -e .

Cloning into 'LlamaFactory'...
remote: Enumerating objects: 636, done.
remote: Counting objects: 100% (636/636), done.
remote: Compressing objects: 100% (477/477), done.
remote: Total 636 (delta 153), reused 417 (delta 99), pack-reused 0 (from 0)
Receiving objects: 100% (636/636), 5.26 MiB | 11.02 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/LlamaFactory
Obtaining file:///content/LlamaFactory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
# template of the llama factory for gemma 3
"ocr_finetune_train":{
    "file_name":"/gdrive/MyDrive/VLMs-Dataset/train_ds-v1-edited.json",
    "formatting":"sharegpt",
    "columns":{
        "messages":"conversations",
        "images":"images"
    }
},
"ocr_finetune_val":{
    "file_name":"/gdrive/MyDrive/VLMs-Dataset/val-v1-edited.json",
    "formatting":"sharegpt",
    "columns":{
        "messages":"conversations",
        "images":"images"
    }
}


#### Notes for ocr_finetune.yaml
- the script is from this file path
/content/LlamaFactory/examples/train_lora/qwen3vl_lora_sft.yaml
- then changing some values depend on the model type and version :       
- model name or path gitting from hugging gace model name .
- Lora rank is between 16 to 128 depending on the model size so if the model is week increase the value to 96, 128 VS if the model is big decrease the number to 16 or 32 .
- dataset & eval_datasetis the name of the keys in the dataset info file .
- template is a gemma3 from llama factory template name .
- cutoff_len is a max tokens of the prompt and the output .
- max_samples is to runa complete iteration of the work in 50 samples to make sure it saves and getting output correctly if it okey with no problems then run it in all samples .
- save_steps is that every 50 step in finetuning make a checkpoint saving .
- eval_steps is being the same number of the save steps to evaluate every checkpoint
- logging_steps is the number to make logging
- learning_rate is the number that helping in the finetuning process and it's not fixed ( examples in llama factory ).
- per_device_train_batch_size : for a small memmory this number must be small .
- gradient_accumulation_steps the number of the steps to update the wights .
- bf16 is to decrease the size of the model in memory without effect the quality .
- report_to is to get a report with value (wandb) weights and biases website

